# 1. Implementez le réseau LeNet, tel que défini dans la quatrième partie du tutoriel, sur les données MNIST (disponible dans torchvision)

In [1]:
import torch
import torchvision
import torchvision.transforms as transforms


transform = transforms.Compose([transforms.ToTensor()])
batch_size = 4

trainset = torchvision.datasets.MNIST(root='./data', train=True,
                                        download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=batch_size,
                                          shuffle=True, num_workers=2)

testset = torchvision.datasets.MNIST(root='./data', train=False,
                                       download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=batch_size,
                                         shuffle=False, num_workers=2)

classes = ('0', '1', '2', '3', '4', '5', '6', '7', '8', '9')

In [2]:
import torch.nn as nn
import torch.nn.functional as F


class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 6, 5)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(6, 16, 5)
        self.fc1 = nn.Linear(16 * 4 * 4, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = torch.flatten(x, 1) # flatten all dimensions except batch
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x


net = Net()

In [3]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(net.parameters(), lr=0.001, momentum=0.9)

In [4]:
for epoch in range(2):  # loop over the dataset multiple times

    running_loss = 0.0
    for i, data in enumerate(trainloader, 0):
        # get the inputs; data is a list of [inputs, labels]
        inputs, labels = data

        # zero the parameter gradients
        optimizer.zero_grad()

        # forward + backward + optimize
        outputs = net(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        # print statistics
        running_loss += loss.item()
        if i % 2000 == 1999:    # print every 2000 mini-batches
            print(f'[{epoch + 1}, {i + 1:5d}] loss: {running_loss / 2000:.3f}')
            running_loss = 0.0

print('Finished Training')

[1,  2000] loss: 1.529
[1,  4000] loss: 0.295
[1,  6000] loss: 0.178
[1,  8000] loss: 0.135
[1, 10000] loss: 0.111
[1, 12000] loss: 0.105
[1, 14000] loss: 0.093
[2,  2000] loss: 0.084
[2,  4000] loss: 0.074
[2,  6000] loss: 0.070
[2,  8000] loss: 0.066
[2, 10000] loss: 0.068
[2, 12000] loss: 0.061
[2, 14000] loss: 0.073
Finished Training


# 2. Calculez la precision (accuracy) et comparez les résultats obtentus avec un MLP

In [5]:
correct = 0
total = 0
correct_pred = {classname: 0 for classname in classes}
total_pred = {classname: 0 for classname in classes}

with torch.no_grad():
    for data in testloader:
        images, labels = data
        outputs = net(images)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

        for label, prediction in zip(labels, predicted):
            if label == prediction:
                correct_pred[classes[label]] += 1
            total_pred[classes[label]] += 1

for classname, correct_count in correct_pred.items():
    accuracy = 100 * float(correct_count) / total_pred[classname]
    print(f'Accuracy for class: {classname:5s} is {accuracy:.1f} %')
print(f'Accuracy of the network on the 10000 test images: {100 * correct / total} %')

Accuracy for class: 0     is 99.4 %
Accuracy for class: 1     is 98.8 %
Accuracy for class: 2     is 98.0 %
Accuracy for class: 3     is 98.5 %
Accuracy for class: 4     is 98.3 %
Accuracy for class: 5     is 97.9 %
Accuracy for class: 6     is 98.9 %
Accuracy for class: 7     is 98.6 %
Accuracy for class: 8     is 98.3 %
Accuracy for class: 9     is 97.8 %
Accuracy of the network on the 10000 test images: 98.44 %


On peut voir que l'on obtient une accuracy de 98.22% avec le CNN. Lors du TP précédent, l'accuracy maximale obtenue avec un MLP était de 98.33%.

# 3. Modifiez le réseau (taille des feature maps, taille du noyau) et commentez les résultats obtenus

In [6]:
class Net2(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 12, 3)  # Augmentation du nombre de feature maps et réduction de la taille du noyau
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(12, 24, 3)  # Augmentation du nombre de feature maps et réduction de la taille du noyau
        self.fc1 = nn.Linear(24 * 5 * 5, 120)  # Ajustement de la taille d'entrée en fonction des nouvelles dimensions
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = torch.flatten(x, 1) # flatten all dimensions except batch
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

In [7]:
net2 = Net2()

import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(net2.parameters(), lr=0.002, momentum=0.9)

for epoch in range(2):  # loop over the dataset multiple times

    running_loss = 0.0
    for i, data in enumerate(trainloader, 0):
        # get the inputs; data is a list of [inputs, labels]
        inputs, labels = data

        # zero the parameter gradients
        optimizer.zero_grad()

        # forward + backward + optimize
        outputs = net2(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        # print statistics
        running_loss += loss.item()
        if i % 2000 == 1999:  # print every 2000 mini-batches
            print(f'[{epoch + 1}, {i + 1:5d}] loss: {running_loss / 2000:.3f}')
            running_loss = 0.0

print('Finished Training')

[1,  2000] loss: 1.084
[1,  4000] loss: 0.218
[1,  6000] loss: 0.151
[1,  8000] loss: 0.123
[1, 10000] loss: 0.105
[1, 12000] loss: 0.095
[1, 14000] loss: 0.087
[2,  2000] loss: 0.063
[2,  4000] loss: 0.059
[2,  6000] loss: 0.070
[2,  8000] loss: 0.057
[2, 10000] loss: 0.055
[2, 12000] loss: 0.060
[2, 14000] loss: 0.054
Finished Training


In [8]:
correct = 0
total = 0
correct_pred = {classname: 0 for classname in classes}
total_pred = {classname: 0 for classname in classes}

with torch.no_grad():
    for data in testloader:
        images, labels = data
        outputs = net2(images)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

        for label, prediction in zip(labels, predicted):
            if label == prediction:
                correct_pred[classes[label]] += 1
            total_pred[classes[label]] += 1

for classname, correct_count in correct_pred.items():
    accuracy = 100 * float(correct_count) / total_pred[classname]
    print(f'Accuracy for class: {classname:5s} is {accuracy:.1f} %')
print(f'Accuracy of the network on the 10000 test images: {100 * correct / total} %')

Accuracy for class: 0     is 98.9 %
Accuracy for class: 1     is 99.3 %
Accuracy for class: 2     is 99.1 %
Accuracy for class: 3     is 99.4 %
Accuracy for class: 4     is 98.7 %
Accuracy for class: 5     is 98.5 %
Accuracy for class: 6     is 98.3 %
Accuracy for class: 7     is 99.2 %
Accuracy for class: 8     is 98.6 %
Accuracy for class: 9     is 98.0 %
Accuracy of the network on the 10000 test images: 98.82 %
